# 05 · Train YOLO11n and evaluate

**Agenda: 75–95 min.** The presenter exports the auto-labeled frames to YOLO format, stages them on Nebius Object Storage, and submits a **Nebius Serverless AI Job** from a notebook (`presenter/train_on_nebius.ipynb`): an L40S GPU container fine-tunes `yolo11n.pt` for 15 epochs and drops `best.pt` back in the bucket.

That model's predictions are on your frames as `yolo11n_preds`, and the evaluation against the auto-labels is the run `eval_yolo`.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F

frames = fo.load_dataset("droid-frames-workshop")
print(frames.list_evaluations())
session = fo.launch_app(frames)

## Model Evaluation panel

Open the **Model Evaluation** panel → `eval_yolo`. mAP per class, confusion matrix, and every cell of the matrix is clickable: click *gripper predicted as robot_arm* and the grid shows those frames.

Or in code:

In [ ]:
results = frames.load_evaluation_results("eval_yolo")
results.print_report()

In [ ]:
# Validation frames only, worst first (most false positives)
val = frames.match(F("split") == "val")
session.view = val.sort_by("eval_yolo_fp", reverse=True)

In [ ]:
# Missed grippers (false negatives) — what does the detector not see?
fn = val.filter_labels("auto_labels", (F("eval_yolo") == "fn") & (F("label") == "gripper"), only_matches=True)
print(len(fn), "val frames with a missed gripper")
session.view = fn

## The job spec, for reference

```python
from presenter.nebius_job import NebiusJob
job = NebiusJob()                                     # Nebius IAM token + Object Storage key
job.upload_dir("build/yolo_export", "runs/stuttgart/input")
job_id = job.submit("stuttgart", "runs/stuttgart", model="yolo11n.pt", epochs=15,
                    platform="gpu-l40s-a", preset="1gpu-8vcpu-32gb")
job.wait(job_id)                                      # PROVISIONING → RUNNING → COMPLETED
job.download("runs/stuttgart/output/best.pt", "presenter/weights/best.pt")
```

The container is public (`ghcr.io/danielgural/fiftyone-yolo-train`) and only needs `INPUT_S3_URI`, `OUTPUT_S3_URI`, `MODEL`, `HYPERPARAMS_JSON`.

## Optional: a 1-epoch fine-tune on your laptop (≈3–5 min CPU, 200 frames)

Same code as the GPU job, tiny scale. `ultralytics` is in `requirements.txt`.

In [ ]:
# OPTIONAL
# from ultralytics import YOLO
# small = frames.match(F("split") == "train").match_tags("near-duplicate", bool=False).take(200, seed=51)
# small.export(export_dir="/tmp/yolo_small", dataset_type=fo.types.YOLOv5Dataset, label_field="auto_labels", split="train",
#              classes=["gripper", "robot_arm", "brick", "scissors", "drawer", "cloth", "box"])
# frames.match(F("split") == "val").take(60, seed=51).export(export_dir="/tmp/yolo_small", dataset_type=fo.types.YOLOv5Dataset,
#              label_field="auto_labels", split="val", classes=["gripper", "robot_arm", "brick", "scissors", "drawer", "cloth", "box"])
# model = YOLO("yolo11n.pt")
# model.train(data="/tmp/yolo_small/dataset.yaml", epochs=1, imgsz=416, batch=16, device="cpu", workers=0, plots=False)
# frames.match(F("split") == "val").take(60, seed=51).apply_model(model, label_field="my_preds")
# session.view = frames.exists("my_preds")

**Next:** predictions on frames are useful. Predictions *on the recording's timeline* are what the robotics team actually wants. Notebook 06.